# TF-IDF + Logistic Regression Baseline

This notebook trains the first baseline model for AI-generated text detection.

We use TF-IDF to convert text into numerical features, then train a Logistic Regression classifier to predict whether the text is human-written or AI-generated.

## Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib
import mlflow
import mlflow.sklearn

## Paths

In [30]:
DATA_DIR = Path("../data/splits")
MODEL_DIR = Path("../models")
OUTPUT_DIR = Path("../outputs")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.parquet"
VAL_FILE = DATA_DIR / "validation.parquet"
TEST_FILE = DATA_DIR / "test.parquet"

## Load datasets

In [31]:
train_df = pd.read_parquet(TRAIN_FILE)
val_df = pd.read_parquet(VAL_FILE)
test_df = pd.read_parquet(TEST_FILE)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (341052, 3)
Validation: (73083, 3)
Test: (73083, 3)


In [32]:
mlflow.set_experiment("AI Text Detection")

2026/07/01 20:20:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/01 20:20:42 INFO mlflow.store.db.utils: Updating database tables
2026/07/01 20:20:45 INFO mlflow.tracking.fluent: Experiment with name 'AI Text Detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Drasti/PROG74040-AI-Text-Detection/notebooks/mlruns/1', creation_time=1782951645252, experiment_id='1', last_update_time=1782951645252, lifecycle_stage='active', name='AI Text Detection', tags={}, trace_location=None, workspace='default'>

## Separate features and labels

In [19]:
X_train = train_df["text"]
y_train = train_df["generated"]

X_val = val_df["text"]
y_val = val_df["generated"]

X_test = test_df["text"]
y_test = test_df["generated"]

## TF-IDF Vectorization

In [20]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=3,
    max_df=0.90
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (341052, 50000)
Validation TF-IDF shape: (73083, 50000)
Test TF-IDF shape: (73083, 50000)


## Train Logistic Regression

In [ ]:
with mlflow.start_run(run_name="TF-IDF Logistic Regression"):

    # Log TF-IDF parameters
    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("max_features", 50000)
    mlflow.log_param("ngram_range", "(1, 2)")
    mlflow.log_param("stop_words", "english")
    mlflow.log_param("min_df", 3)
    mlflow.log_param("max_df", 0.90)

    # Create and train model
    log_reg = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)

    log_reg.fit(X_train_tfidf, y_train)

    # Validation evaluation
    val_preds = log_reg.predict(X_val_tfidf)
    val_probs = log_reg.predict_proba(X_val_tfidf)[:, 1]

    val_accuracy = accuracy_score(y_val, val_preds)
    val_precision = precision_score(y_val, val_preds)
    val_recall = recall_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds)
    val_roc_auc = roc_auc_score(y_val, val_probs)

    print("Validation Accuracy:", val_accuracy)
    print("Validation Precision:", val_precision)
    print("Validation Recall:", val_recall)
    print("Validation F1:", val_f1)
    print("Validation ROC-AUC:", val_roc_auc)

    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("val_precision", val_precision)
    mlflow.log_metric("val_recall", val_recall)
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_roc_auc", val_roc_auc)

    print("\nValidation Classification Report:")
    print(classification_report(
        y_val,
        val_preds,
        target_names=["Human", "AI"]
    ))

    # Test evaluation
    test_preds = log_reg.predict(X_test_tfidf)
    test_probs = log_reg.predict_proba(X_test_tfidf)[:, 1]

    test_results = {
        "accuracy": accuracy_score(y_test, test_preds),
        "precision": precision_score(y_test, test_preds),
        "recall": recall_score(y_test, test_preds),
        "f1_score": f1_score(y_test, test_preds),
        "roc_auc": roc_auc_score(y_test, test_probs)
    }

    print("\nTest Results:")
    print(test_results)

    mlflow.log_metric("test_accuracy", test_results["accuracy"])
    mlflow.log_metric("test_precision", test_results["precision"])
    mlflow.log_metric("test_recall", test_results["recall"])
    mlflow.log_metric("test_f1", test_results["f1_score"])
    mlflow.log_metric("test_roc_auc", test_results["roc_auc"])

    # Save model and vectorizer locally
    model_path = MODEL_DIR / "tfidf_logistic_regression_model.joblib"
    vectorizer_path = MODEL_DIR / "tfidf_vectorizer.joblib"

    joblib.dump(log_reg, model_path)
    joblib.dump(tfidf, vectorizer_path)

    print("\nModel and vectorizer saved successfully.")

    # Log model to MLflow
    mlflow.sklearn.log_model(log_reg, "logistic_regression_model")

    # Log saved files as MLflow artifacts
    mlflow.log_artifact(str(model_path))
    mlflow.log_artifact(str(vectorizer_path))

    # Save results CSV
    results_df = pd.DataFrame([{
        "model": "TF-IDF + Logistic Regression",
        "accuracy": test_results["accuracy"],
        "precision": test_results["precision"],
        "recall": test_results["recall"],
        "f1_score": test_results["f1_score"],
        "roc_auc": test_results["roc_auc"]
    }])

    results_path = OUTPUT_DIR / "tfidf_logistic_regression_results.csv"
    results_df.to_csv(results_path, index=False)

    mlflow.log_artifact(str(results_path))

    results_df

c:\Users\Drasti\anaconda3\envs\aimlcourse\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Drasti\anaconda3\envs\aimlcourse\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


NameError: name 'roc_auc_score' is not defined